# 🧠 EEG/BCI Research Copilot — Google Colab Demo

This notebook runs the full EEG/BCI Research Copilot in Google Colab.
It installs dependencies, clones the repo, and launches the Gradio UI.

**Before running:** Add your Gemini API key in Step 2.
Get a free key at: https://aistudio.google.com

## Step 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install langchain==0.2.16 langchain-google-genai==1.0.10 \
             langchain-community==0.2.16 langchain-chroma==0.1.4 \
             chromadb==0.5.5 sentence-transformers==3.0.1 \
             pymupdf==1.24.9 gradio==4.42.0 python-dotenv==1.0.1 \
             tiktoken==0.7.0 -q

print('✅ Dependencies installed')

## Step 2: Clone Repository and Set API Key

In [ ]:
import os

# Clone your repository
!git clone https://github.com/khunsa123/eeg-bci-copilot.git
%cd eeg-bci-copilot

# Set your Gemini API key
# Get free key at: https://aistudio.google.com
GEMINI_API_KEY = "your_gemini_api_key_here"  # <-- REPLACE THIS

# Write to .env file
with open('.env', 'w') as f:
    f.write(f'GEMINI_API_KEY={GEMINI_API_KEY}\n')

# Create required directories
os.makedirs('data/papers', exist_ok=True)
os.makedirs('data/chroma_db', exist_ok=True)

print('✅ Repository ready')

## Step 3: Upload a Test Paper
Upload a PDF to test the system.

In [ ]:
from google.colab import files
import shutil

print('📤 Upload your EEG/neuroscience PDF papers...')
uploaded = files.upload()

# Move uploaded files to data/papers/
for filename in uploaded.keys():
    shutil.move(filename, f'data/papers/{filename}')
    print(f'  ✅ Moved: {filename} → data/papers/')

## Step 4: Ingest Papers

In [ ]:
import sys
sys.path.append('src')

from ingest import ingest_pdfs

chunks = ingest_pdfs()
print(f'\n✅ Ingestion complete: {chunks} chunks stored in ChromaDB')

## Step 5: Test Retrieval

In [ ]:
from retriever import Retriever

retriever = Retriever()

# List papers
papers = retriever.list_papers()
print('📚 Papers in database:')
for p in papers:
    print(f'  - {p}')

# Test search
print('\n🔍 Test search: EEG preprocessing pipeline')
results = retriever.search('EEG preprocessing pipeline artifact removal')
for r in results[:2]:
    print(f'\n  [{r["filename"]} p.{r["page"]}] Score: {r["score"]}')
    print(f'  {r["text"][:300]}...')

## Step 6: Launch Gradio App
This generates a public URL you can share.

In [ ]:
# Launch with share=True to get a public URL
!python app.py --share

## Alternative: Use Agent Directly in Notebook

In [ ]:
from agent import CopilotAgent

agent = CopilotAgent()

# Example queries
queries = [
    "List all papers in the database",
    "What EEG preprocessing methods are mentioned in the papers?",
    "Suggest a pipeline for EEG emotion recognition",
]

for query in queries:
    print(f'\n🔬 Query: {query}')
    print('─' * 60)
    response = agent.chat(query)
    print(response)
    print()